In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Analyze accuracy values from a mlm accuracy file and produce:
1. Summary statistics (mean, median, IQR, outlier bounds).
2. A boxplot saved as a PNG.

Each line of the input file is expected to contain a single numeric value.
Non-numeric lines are ignored.
"""

import argparse
import os
import numpy as np
import matplotlib.pyplot as plt


def load_accuracy_values(path: str) -> list[float]:
    """Load numeric accuracy values from a text file, skipping non-numeric lines."""
    values: list[float] = []
    with open(path, "r") as f:
        for line in f:
            try:
                num = float(line.strip())
                if not np.isnan(num):
                    values.append(num)
            except ValueError:
                # Ignore lines that cannot be converted to float
                continue
    return values


def compute_stats(values: list[float]) -> dict:
    """Compute basic statistics and IQR-based outlier bounds."""
    if not values:
        raise ValueError("No valid numeric values found in input file.")

    arr = np.asarray(values, dtype=float)

    stats = {
        "mean": float(np.mean(arr)),
        "median": float(np.median(arr)),
        "Q1": float(np.percentile(arr, 25)),
        "Q3": float(np.percentile(arr, 75)),
    }
    stats["IQR"] = stats["Q3"] - stats["Q1"]
    stats["lower_bound"] = stats["Q1"] - 1.5 * stats["IQR"]
    stats["upper_bound"] = stats["Q3"] + 1.5 * stats["IQR"]

    return stats


def save_stats(stats: dict, out_path: str) -> None:
    """Write stats to a text file."""
    lines = [
        f"Mean accuracy: {stats['mean']}",
        f"Median accuracy: {stats['median']}",
        f"Q1: {stats['Q1']}",
        f"Q3: {stats['Q3']}",
        f"IQR: {stats['IQR']}",
        f"Lower Bound: {stats['lower_bound']}",
        f"Upper Bound: {stats['upper_bound']}",
    ]
    with open(out_path, "w") as f:
        f.write("\n".join(lines))


def save_boxplot(values: list[float], out_path: str, title: str) -> None:
    """Save a boxplot of accuracy values as PNG."""
    plt.figure(figsize=(10, 5))
    plt.boxplot(values)
    plt.ylim(0.0, 1.0)  # assuming accuracy in [0, 1]
    plt.xlabel("Pretrain Accuracy")
    plt.ylabel("Values")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Analyze accuracy values and generate stats + boxplot."
    )
    parser.add_argument(
        "--input",
        required=True,
        help="Path to the input text file containing one accuracy value per line.",
    )
    parser.add_argument(
        "--output-dir",
        required=True,
        help="Directory where the stats text file and boxplot PNG will be saved.",
    )
    parser.add_argument(
        "--prefix",
        default="accuracy",
        help="Prefix for the output filenames (default: 'accuracy').",
    )
    parser.add_argument(
        "--title",
        default="Accuracy Distribution",
        help="Title to display on the boxplot.",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()

    os.makedirs(args.output_dir, exist_ok=True)

    values = load_accuracy_values(args.input)
    stats = compute_stats(values)

    stats_path = os.path.join(args.output_dir, f"{args.prefix}_stats.txt")
    plot_path = os.path.join(args.output_dir, f"{args.prefix}_boxplot.png")

    save_stats(stats, stats_path)
    save_boxplot(values, plot_path, args.title)

    print(f"Stats saved to: {stats_path}")
    print(f"Boxplot saved to: {plot_path}")


if __name__ == "__main__":
    main()
